In [1]:
"""HipMRI dataset discovery and patient split checks."""

from pathlib import Path
import re


# Extract the patient ID while checking the expected filename structure.
IMAGE_PATTERN = re.compile(
    r"case_(\d+)_week_\d+_slice_\d+\.nii\.gz"
)
SPLITS = ("train", "validate", "test")


def find_pairs(data_dir, split):
    """Return matching image and mask paths for one supplied split."""
    if split not in SPLITS:
        raise ValueError(f"Unknown split: {split}")

    data_dir = Path(data_dir)
    image_dir = data_dir / f"keras_slices_{split}"
    mask_dir = data_dir / f"keras_slices_seg_{split}"

    # Catch incorrect paths before attempting to load any images.
    for folder in (image_dir, mask_dir):
        if not folder.is_dir():
            raise FileNotFoundError(f"Dataset folder not found: {folder}")

    images = sorted(image_dir.glob("*.nii.gz"))
    if not images:
        raise ValueError(f"No NIfTI images found in {image_dir}")

    pairs = []
    expected_masks = set()

    for image_path in images:
        if IMAGE_PATTERN.fullmatch(image_path.name) is None:
            raise ValueError(f"Unexpected image filename: {image_path.name}")

        # Pair by filename rather than relying on directory ordering.
        mask_name = image_path.name.replace("case_", "seg_", 1)
        mask_path = mask_dir / mask_name

        if not mask_path.is_file():
            raise FileNotFoundError(f"Missing mask: {mask_path}")

        pairs.append((image_path, mask_path))
        expected_masks.add(mask_name)

    # Also detect masks that have no corresponding image.
    actual_masks = {p.name for p in mask_dir.glob("*.nii.gz")}
    extra_masks = actual_masks - expected_masks
    if extra_masks:
        raise ValueError(
            f"{split}: {len(extra_masks)} masks have no matching image"
        )

    return pairs


def patient_ids(pairs):
    """Read patient IDs from the verified image filenames."""
    return {
        IMAGE_PATTERN.fullmatch(image.name).group(1)
        for image, _ in pairs
    }


def audit_splits(data_dir):
    """Check all supplied splits and reject overlapping patient IDs."""
    splits = {split: find_pairs(data_dir, split) for split in SPLITS}
    patients = {
        split: patient_ids(pairs) for split, pairs in splits.items()
    }

    for first, second in (
        ("train", "validate"),
        ("train", "test"),
        ("validate", "test"),
    ):
        overlap = patients[first] & patients[second]
        if overlap:
            raise ValueError(
                f"Patient overlap between {first} and {second}: "
                f"{sorted(overlap)}"
            )

    print(f"{'Split':<12} {'Pairs':>8} {'Patients':>10}")
    for split in SPLITS:
        print(
            f"{split:<12} {len(splits[split]):>8} "
            f"{len(patients[split]):>10}"
        )

    print("All image–mask pairs matched; no patient overlap.")
    return splits


# Use the dataset's absolute location while developing in a notebook.
data_dir = Path(
    "/lambda/nfs/comp3710-virginia/PatternAnalysis-2026/"
    "recognition/ResidualUNet_s4749642/data/keras_slices_data"
)

splits = audit_splits(data_dir)

Split           Pairs   Patients
train           11460         31
validate          660          4
test              540          3
All image–mask pairs matched; no patient overlap.


In [ ]:
# %pip install nibabel
# Anatomical mapping confirmed through consultation with a doctor.
LABEL_NAMES = {
    0: "Outside body",
    1: "Remaining tissue",
    2: "Bone",
    3: "Bladder",
    4: "Rectum",
    5: "Prostate",
}

NUM_CLASSES = len(LABEL_NAMES)
PROSTATE_LABEL = 5

In [ ]:
import nibabel as nib
import numpy as np

# Sanity test - make sure all of the image pairs actually work
def load_pair(image_path, mask_path):
    """Load and validate one 2D MRI–mask pair."""
    image_nii = nib.load(image_path)
    mask_nii = nib.load(mask_path)

    # Remove singleton dimensions, keeping the stored orientation.
    image = np.squeeze(image_nii.get_fdata(dtype=np.float32))
    mask = np.squeeze(mask_nii.get_fdata())

    if image.ndim != 2 or mask.ndim != 2:
        raise ValueError("Expected 2D image and mask")

    if image.shape != mask.shape:
        raise ValueError("Image and mask dimensions differ")

    # Check spatial metadata and numerical validity before preprocessing.
    if not np.allclose(image_nii.affine, mask_nii.affine):
        raise ValueError("Image and mask affines differ")

    if not np.isfinite(image).all() or not np.isfinite(mask).all():
        raise ValueError("Image or mask contains NaN or infinite values")

    if not np.allclose(mask, np.round(mask)) or mask.min() < 0:
        raise ValueError("Mask must contain non-negative integer labels")

    # Standardise MRI intensities; leave segmentation labels unchanged.
    image = (image - image.mean()) / (image.std() + 1e-8)
    mask = np.round(mask).astype(np.int64)

    return image, mask

In [ ]:
# Load the first verified training pair.
image_path, mask_path = splits["train"][0]
image, mask = load_pair(image_path, mask_path)

print("Example:", image_path.name)
print("Image:", image.shape, image.dtype)
print("Mask:", mask.shape, mask.dtype)
print("Normalised image mean:", round(float(image.mean()), 4))
print("Normalised image standard deviation:", round(float(image.std()), 4))
print("Mask labels:", np.unique(mask))

In [ ]:
# Search the dataset and its parent folder for label documentation.
extensions = {".txt", ".md", ".json", ".csv", ".yaml", ".yml", ".xml"}
documents = []

for path in data_dir.parent.rglob("*"):
    if path.is_file() and (
        path.suffix.lower() in extensions
        or "readme" in path.name.lower()
    ):
        documents.append(path)

print("Documentation files:")
for path in sorted(documents):
    print(path.relative_to(data_dir.parent))

if not documents:
    print("No documentation found beside the uploaded dataset.")

In [ ]:
# Some NIfTI files include descriptions or extensions with label information.
mask_nii = nib.load(splits["train"][0][1])

print("Description:", mask_nii.header["descrip"].item())
print("Auxiliary information:", mask_nii.header["aux_file"].item())
print("Extensions:", mask_nii.header.extensions)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F


class HipMRIDataset(Dataset):
    """Load paired slices and pad them to a common size."""

    def __init__(self, pairs, target_size=(256, 144)):
        self.pairs = list(pairs)
        self.target_size = target_size

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):
        image_path, mask_path = self.pairs[index]
        image, mask = load_pair(image_path, mask_path)

        image = torch.from_numpy(
            np.ascontiguousarray(image)
        ).unsqueeze(0)
        mask = torch.from_numpy(np.ascontiguousarray(mask))

        height, width = mask.shape
        target_h, target_w = self.target_size

        # Stop rather than silently crop a larger image.
        if height > target_h or width > target_w:
            raise ValueError(
                f"{image_path.name}: shape {(height, width)} exceeds "
                f"target {self.target_size}"
            )

        # Apply identical centred padding to the image and mask.
        pad_h, pad_w = target_h - height, target_w - width
        padding = (
            pad_w // 2, pad_w - pad_w // 2,
            pad_h // 2, pad_h - pad_h // 2,
        )

        # Zero is the mean intensity after normalisation.
        image = F.pad(image, padding, value=0)

        # Mark artificial pixels so the loss ignores them.
        mask = F.pad(mask, padding, value=-100)

        return image, mask

In [ ]:
train_dataset = HipMRIDataset(splits["train"])

# Start with zero worker processes for straightforward notebook debugging.
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
)

images, masks = next(iter(train_loader))

print("Training slices:", len(train_dataset))
print("Image batch:", tuple(images.shape), images.dtype)
print("Mask batch:", tuple(masks.shape), masks.dtype)
print("Labels in this batch:", torch.unique(masks).tolist())

In [ ]:
import matplotlib.pyplot as plt

# Find an example containing all six numeric labels.
for image_path, mask_path in splits["train"]:
    image, mask = load_pair(image_path, mask_path)
    if set(np.unique(mask)) == set(range(6)):
        break
else:
    raise ValueError("No training slice containing all six labels was found.")

fig, axes = plt.subplots(2, 4, figsize=(14, 9))

axes[0, 0].imshow(image, cmap="gray")
axes[0, 0].set_title("MRI")

# Display the complete reference mask with fixed label colours.
axes[0, 1].imshow(mask, cmap="tab10", vmin=0, vmax=9)
axes[0, 1].set_title("All labels")

# Show each label separately to make its anatomical position clearer.
for label, ax in zip(range(6), axes.flat[2:]):
    ax.imshow(image, cmap="gray")
    ax.imshow(
        np.ma.masked_where(mask != label, mask),
        cmap="tab10",
        vmin=0,
        vmax=9,
        interpolation="nearest",
        alpha=0.5,
    )
    ax.set_title(f"Label {label}")

for ax in axes.flat:
    ax.axis("off")

plt.suptitle(image_path.name)
plt.tight_layout()
plt.show()


In [ ]:
# Labels
# 0 (Blue): Outside body
# 1 (Orange): Remaining tissue
# 2 (Green): Bone
# 3 (Red): Bladder
# 4 (Purple): Rectum
# 5 (Brown: Prostate

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Select only this patient and scan, then sort slices numerically.
case_pairs = [
    (image_path, mask_path)
    for image_path, mask_path in splits["train"]
    if image_path.name.startswith("case_004_week_0_slice_")
]
case_pairs.sort(
    key=lambda pair: int(pair[0].name.split("_slice_")[1].split(".")[0])
)

# Set to 1 to show every slice, or increase for a shorter overview.
slice_step = 5
selected = case_pairs[::slice_step]
assert selected, "No matching slices found."

columns = 5
rows = (len(selected) + columns - 1) // columns
fig, axes = plt.subplots(
    rows, columns, figsize=(15, rows * 4), squeeze=False
)

for ax, (image_path, mask_path) in zip(axes.flat, selected):
    image, mask = load_pair(image_path, mask_path)
    slice_number = image_path.name.split("_slice_")[1].split(".")[0]

    ax.imshow(image, cmap="gray")

    # Keep label 0 transparent and use consistent colours across slices.
    ax.imshow(
        np.ma.masked_where(mask == 0, mask),
        cmap="tab10",
        vmin=0,
        vmax=9,
        alpha=0.4,
        interpolation="nearest",
    )

    ax.set_title(f"Slice {slice_number}")
    ax.axis("off")

# Hide unused panels at the end of the grid.
for ax in list(axes.flat)[len(selected):]:
    ax.axis("off")

# Use numeric labels until anatomical meanings are confirmed.
cmap = plt.get_cmap("tab10")
legend = [
    Patch(facecolor=cmap(label), label=f"Label {label}")
    for label in range(1, 6)
]
fig.legend(handles=legend, loc="lower center", ncol=5)

fig.suptitle("Patient 004, week 0: ground-truth overlays")
plt.tight_layout(rect=[0, 0.04, 1, 0.97])
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Select patient 004, week 0, and sort by numeric slice number.
case_pairs = [
    pair for pair in splits["train"]
    if pair[0].name.startswith("case_004_week_0_slice_")
]
case_pairs.sort(
    key=lambda pair: int(pair[0].name.split("_slice_")[1].split(".")[0])
)

# Show every fifth slice; change to 1 to display all slices.
selected = case_pairs[::5]
assert selected, "No matching slices found."

columns = 5
rows = (len(selected) + columns - 1) // columns
fig, axes = plt.subplots(
    rows, columns, figsize=(15, rows * 4), squeeze=False
)

for ax, (image_path, _) in zip(axes.flat, selected):
    # Load MRI intensities directly, without normalisation or masks.
    image = np.squeeze(nib.load(image_path).get_fdata())
    slice_number = image_path.name.split("_slice_")[1].split(".")[0]

    ax.imshow(image, cmap="gray")
    ax.set_title(f"Slice {slice_number}")
    ax.axis("off")

# Hide unused panels.
for ax in list(axes.flat)[len(selected):]:
    ax.axis("off")

fig.suptitle("Patient 004, week 0: MRI slices")
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()